In [9]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial.transform import Rotation as R

class codebooks():
    def dft(self, dim):
        seq = np.matrix(np.arange(dim))
        mat = seq.conj().T * seq
        w = np.exp(-1j * 2 * np.pi * mat / dim)
        return w

    def dft_upa(self, rows, cols):
        # DFT matrices for rows and columns
        w_row = self.dft(rows)
        w_col = self.dft(cols)
        # Create 2D codebook by outer product
        upa_codebook = np.kron(w_row, w_col)  # Kronecker product to create 2D beams
        return upa_codebook

class patterns(): # theta = zenith, phi = azimuth
    def isotropic(self, theta, phi):
        return 1
    def dipole(self, theta, phi):
        return 1.67*np.sin(theta)**3

class utils():
    def __init__(self, alpha, beta, gamma, pattern_fn):
        self.pattern_fn = pattern_fn
        # extrínseca: z depois y depois x (mesma ordem efetiva do seu Rx@Ry@Rz)
        self.rot = R.from_euler('zyx', [alpha, beta, gamma], degrees=False)

    def rotation3D(self, theta, phi):
        # mundo: (theta=zenith, phi=azimute)
        x = np.sin(theta) * np.cos(phi)
        y = np.sin(theta) * np.sin(phi)
        z = np.cos(theta)

        v_world = np.stack([x, y, z], axis=-1) # (..., 3)

        # mundo -> corpo (rotação inversa)
        v_body = self.rot.apply(v_world, inverse=True) # (..., 3)

        xb, yb, zb = v_body[..., 0], v_body[..., 1], v_body[..., 2]
        theta_b = np.arccos(np.clip(zb, -1.0, 1.0))
        phi_b = np.arctan2(yb, xb) % (2*np.pi)

        return self.pattern_fn(theta_b, phi_b)

class AntennaArray:
    def __init__(self, XYZ=[1,1,1], d_XYZ=[0.5,0.5,0.5]):
        self.XYZ = XYZ # Linhas e colunas da UPA
        self.d_XYZ = d_XYZ  # Espaçamento (lambda/2)

    def gridDefault(self, rev_theta_phi=[[0,181,1],[0,361,1]]):
        deg2rad = np.pi/180
        self.theta = np.arange(*rev_theta_phi[0]) * deg2rad 
        self.phi = np.arange(*rev_theta_phi[1]) * deg2rad
        self.Theta, self.Phi = np.meshgrid(self.theta, self.phi)

    def gridCart2Sph(self, WHFvyFvx=[480, 640, 42, 69]):
        '''
        Height, Width, fov_x_deg, fov_y_deg
        Pixels altura, largura, Fov horizontal e vertical da camera 
        '''
        H, W, fov_y_deg, fov_x_deg  = WHFvyFvx

        self.phi = [a for a in range(H)]
        self.theta = [a for a in range(W)]
        
        # Exemplo de FOVs (ajuste para a sua câmera)
        fov_x = np.deg2rad(fov_x_deg) # campo de visão horizontal
        fov_y = np.deg2rad(fov_y_deg) # campo de visão vertical
        # Centro ótico (assumido no centro da imagem)
        cx = W / 2.0
        cy = H / 2.0
        # Focais em pixels (modelo pinhole simples)
        fx = W / (2.0 * np.tan(fov_x / 2.0))
        fy = H / (2.0 * np.tan(fov_y / 2.0))
        
        # Grade de coordenadas de pixel (u, v)
        # u: colunas (0 .. W-1) -> eixo X da imagem
        # v: linhas  (0 .. H-1) -> eixo Y da imagem (para baixo)
        u, v = np.meshgrid(np.arange(W), np.arange(H))

        # Converte para coordenadas normalizadas na câmera
        # x: direita, y: para cima, z: pra frente
        x = (u - cx) / fx
        y = -(v - cy) / fy   # sinal negativo pq v cresce pra baixo na imagem
        z = np.ones_like(x)  # plano z = 1 (direção para frente)

        # Vetor direção (X, Y, Z)
        # (a normalização não é estritamente necessária pros ângulos,
        #  mas deixa o vetor unitário e ajuda na interpretação)
        r_norm = np.sqrt(x**2 + y**2 + z**2)
        X = x / r_norm
        Y = y / r_norm
        Z = z / r_norm

        # Azimute phi: ângulo no plano horizontal (X-Z), em torno do eixo Y
        # 0 rad -> frente (Z+), positivo -> gira para X+ (direita)
        self.Phi = np.arctan2(X, Z)

        # Elevação theta: ângulo vertical em relação ao plano horizontal
        # 0 rad -> horizonte, positivo -> cima (Y+), negativo -> baixo
        # theta = atan2(Y, sqrt(X^2 + Z^2))
        self.Theta = np.arccos(Y) 

    def ArrayFactor(self, codeword, pattern): # array pattern (funcion of codeword (steering vector) and antenna pattern)
        self.AF = np.zeros((len(self.phi), len(self.theta)), dtype=complex)     # grid de angulos de interesse (phi, theta)
        for i in range(len(self.phi)):                                          
            for j in range(len(self.theta)):
                t, p = self.Theta[i, j], self.Phi[i, j]
                steering_vector = np.zeros(self.XYZ[0] * self.XYZ[1] * self.XYZ[2], dtype=complex) # Vetor 1D das antenas (flattened)
                for x in range(self.XYZ[0]):
                    for y in range(self.XYZ[1]):
                        for z in range(self.XYZ[2]):
                            idx = x * self.XYZ[1] * self.XYZ[2] + y * self.XYZ[2] + z
                            steering_vector[idx] = np.exp(1j * 2 * np.pi * ( # positive expands to 0 phi    # 
                                x * self.d_XYZ[0] * np.sin(t) * np.cos(p) 
                                + y * self.d_XYZ[1] * np.sin(t) * np.sin(p)
                                + z * self.d_XYZ[2] * np.cos(t)
                            ))
                self.AF[i, j] = np.sum(codeword * steering_vector) * pattern(t, p)

    def plot3D(self):
        total_antennas= self.XYZ[0] * self.XYZ[1] * self.XYZ[2]
        AF_mag = np.abs(self.AF)
        #AF_mag = (AF_mag - np.min(AF_mag)) / (np.max(AF_mag) - np.min(AF_mag))  # min-max normalization

        # Coordenadas esféricas para cartesianas
        X = AF_mag * np.sin(self.Theta) * np.cos(self.Phi)
        Y = AF_mag * np.sin(self.Theta) * np.sin(self.Phi)
        Z = AF_mag * np.cos(self.Theta)

        x_range = [-total_antennas*3, total_antennas*3]
        y_range = [-total_antennas*3, total_antennas*3]
        z_range = [-total_antennas*3, total_antennas*3]

        fig = go.Figure(data=[
            go.Surface(
                x=X,
                y=Y,
                z=Z,
                surfacecolor=AF_mag,
                colorscale='ice',
                colorbar=dict(title='Magnitude')
            )
        ])

        fig.update_layout(
            title=f'Padrão de Radiação',
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z',
                aspectmode='cube',
                xaxis=dict(range=x_range),
                yaxis=dict(range=y_range),
                zaxis=dict(range=z_range)
            ),
            width=800,
            height=600
        )
        fig.show()
        return
    
    def plot2D(self):
        azimuthal_cut = np.abs(self.AF[:, 90])
        fig = go.Figure(data=[
            go.Scatterpolar(
                r=azimuthal_cut,
                theta=self.phi/(np.pi/180),
                mode='lines',
                line=dict(color='black', width=1),
                name='Corte Azimutal (θ = 90°)'
            )
        ])
        fig.update_layout(
            width=600,
            height=600
        )
        fig.show()
        return

    def plotCart2Sph(self):
        # Exemplo: sua matriz 100x100 (substitua por sua matriz real)
        AF_mag = np.abs(self.AF)
        matriz = AF_mag   # ou qualquer matriz 2D que você tenha

        plt.figure(figsize=(8,6))
        plt.imshow(matriz, cmap='gray', interpolation='nearest')

        # Opcional: colocar os eixos em graus (azimuth e elevation)
        az = np.rad2deg(np.arctan(np.linspace(-0.5,0.5,matriz.shape[1]) * np.tan(69/2 * np.pi/180)))
        el = np.rad2deg(np.arccos(np.linspace(-0.5,0.5,matriz.shape[0]) * np.cos(42/2 * np.pi/180)))

        plt.xticks(np.linspace(0, matriz.shape[1]-1, 9), np.round(np.linspace(az[0], az[-1], 9),1))
        plt.yticks(np.linspace(0, matriz.shape[0]-1, 7), np.round(np.linspace(el[0], el[-1], 7),1))

        plt.xlabel('Azimuth (°)')
        plt.ylabel('Elevation (°)')
        plt.title('Radiation Pattern - Grayscale')
        plt.colorbar(label='Ganho (dB)')       # ajuste o range de cores conforme necessário
        plt.show()
    
        return AF_mag
    
    # dipolo cruzado

# Gerar as Mascaras para cada codeword e sobrepor em imagem, normalizar


In [ ]:
# array dimensions # "2D"
x,y,z = 4,1,1
# Object array receive: array dimensions and inter element spacing
AA = AntennaArray(XYZ=[x,y,z], d_XYZ=[0.5,0.5,0.5])
# Create a theta phi grid with 1 degree resolution
AA.gridDefault(rev_theta_phi=[[0,181,1],[0,361,1]])
# create a codebook object
CB = codebooks()
# Create codebook for planar array dimension (2d Case)
cw = CB.dft_upa(4, 1)
cd_idx = 0
cw = cw[:,cd_idx].A1 # A1 flatten matrix

# create pattern object
ptr = patterns()
# defines
u = utils(0, 0, 0, ptr.isotropic)
AA.ArrayFactor(cw, u.rotation3D)
#AA.ArrayFactor(cw, ptr.dipole)
AA.plot3D()
#AA.plotCart2Sph()

In [3]:
x,y,z = 2,2,1
AA = AntennaArray(XYZ=[x,y,z], d_XYZ=[0.5,0.5,0.5])
AA.gridDefault(rev_theta_phi=[[0,181,1],[0,361,1]])
#AA.gridCart2Sph([480, 640, 42, 69])
CB = codebooks()
cd_idx = 0
cw = CB.dft_upa(2, 2)
cw = cw[:,cd_idx].A1
ptr = patterns()
AA.ArrayFactor(cw, ptr.isotropic)
AA.plot3D()
#AA.plotCart2Sph()

In [19]:
x,y,z = 4,4,1
AA = AntennaArray(XYZ=[x,y,z], d_XYZ=[0.5,0.5,0.5])
AA.gridDefault(rev_theta_phi=[[0,181,1],[0,361,1]])
#AA.gridCart2Sph([480, 640, 42, 69])
CB = codebooks()
cd_idx = 8
cw = CB.dft_upa(4, 4)
cw = cw[:,cd_idx].A1
ptr = patterns()
AA.ArrayFactor(cw, ptr.dipole)
AA.plot3D()
#AA.plotCart2Sph()

In [ ]:
#mikrotik = np.load('./default_mikrotik_cb.npy')
#print(np.shape(mikrotik[:,0]))
#print((mikrotik[:,0]))

masks=[]
x,y,z = 1,6,6
AA = AntennaArray(XYZ=[x,y,z], d_XYZ=[0.5,0.5,0.5])
#AA.gridDefault(rev_theta_phi=[[0,181,1],[0,361,1]])
AA.gridCart2Sph([480, 640, 42, 69])
cw = np.load('./default_mikrotik_cb.npy')
ptr = patterns()
u = utils(0, 0, 0, ptr.isotropic)
for cd_idx in range(64):
    cwo = cw[:,cd_idx]
    AA.ArrayFactor(cwo, u.rotation3D)
    #AA.plot3D()
    mask = AA.plotCart2Sph()
    masks.append(mask)
np.save('masksISO.npy', masks, allow_pickle=True)

#lista_carregada = np.load('masks.npy', allow_pickle=True)
# flipar blilateralmente pois estamos olhando de frente para o padrão, não a partir dele